# Step 1 · Prepare inputs (AD and MCI)

Builds the inputs shared by **every** model, so all models use identical participants, outcomes,
folds and evaluation times.

| Part | What | Helper (`.py`) | Output |
|---|---|---|---|
| 1a | Endpoint-safe longitudinal data from the raw NACC/AIBL files (rebuilt only if raw data or code change) | `prepare_history_information.py` | `results/preprocessing/{AD,MCI}/` |
| 1b | Landmark cohort, outer folds, visits up to month 48, NACC development history, evaluation grid | `export_landmark_inputs.py`, `landmark_common.py` | `results/{AD,MCI}/inputs/` |

Inclusion: event-free at month 48, at least 3 visits by month 48, follow-up beyond month 48.
Checks of the pipeline (smoke test, validation scripts) are in `code/checks/`.

Next: `02_python_models_{AD,MCI}.ipynb`, then `04_evaluate_and_plot.ipynb`.

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_DIR = Path(os.environ.get("SURVIVAL_PROJECT_DIR", "/Users/yihan/Documents/Github/survival_Anlysis_Clustering"))
if str(PROJECT_DIR / "code") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "code"))
import landmark_common as lc
from export_landmark_inputs import export

MODE = "standard"

## 1a · Endpoint-safe longitudinal data

In [2]:
data_root = lc.longitudinal_dir("AD").parent
print("Longitudinal inputs:", data_root)
display(pd.read_csv(data_root / "cohort_summary.csv"))

Longitudinal inputs: /Users/yihan/Documents/Github/survival_Anlysis_Clustering/results/preprocessing


,cohort,endpoint,source_participants,usable_baseline_participants,landmark_eligible,events_after_48_by_108,competing_stops_after_48
0,NACC,AD,50785,12889,5351,613,0
1,AIBL,AD,2505,1497,914,25,0
2,NACC,MCI,50785,8735,3978,498,180
3,AIBL,MCI,2505,1241,804,64,9


## 1b · Shared landmark inputs

In [3]:
summaries = []
for endpoint in lc.ENDPOINTS:
    summary = export(endpoint, MODE)
    summary.to_csv(lc.inputs_dir(endpoint, MODE) / "cohort_summary.csv", index=False)
    summaries.append(summary)
    print(f"{endpoint} -> {lc.inputs_dir(endpoint, MODE)}")
display(pd.concat(summaries, ignore_index=True))

AD [standard] -> /Users/yihan/Documents/Github/survival_Anlysis_Clustering/results/AD/inputs


MCI [standard] -> /Users/yihan/Documents/Github/survival_Anlysis_Clustering/results/MCI/inputs


AD [smoke] -> /Users/yihan/Documents/Github/survival_Anlysis_Clustering/results/smoke/AD/inputs


MCI [smoke] -> /Users/yihan/Documents/Github/survival_Anlysis_Clustering/results/smoke/MCI/inputs


,cohort,participants,events,events_by_24m,events_by_36m,events_by_60m,median_visits_by_48,endpoint,run_mode,outer_folds
0,NACC,5351,896,311,421,613,4.0,AD,standard,3
1,AIBL,914,41,8,13,25,3.0,AD,standard,3
2,NACC,3978,805,222,310,498,4.0,MCI,standard,3
3,AIBL,804,81,32,52,64,3.0,MCI,standard,3
4,NACC,5351,896,311,421,613,4.0,AD,smoke,2
5,AIBL,914,41,8,13,25,3.0,AD,smoke,2
6,NACC,3978,805,222,310,498,4.0,MCI,smoke,2
7,AIBL,804,81,32,52,64,3.0,MCI,smoke,2
